In [65]:
# !git clone https://github.com/had-A42/Actions-Speak-Louder-than-Words-research.git
# %cd Actions-Speak-Louder-than-Words-research
# !git checkout miftakhutdinov
# # !git pull origin miftakhutdinov

In [66]:
!pip3 install -r requirements.txt

  Cloning https://github.com/evfro/polara.git to /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-u4mjxjc2/polara_65159b1df5d349e4933241282b5f60d9
  Running command git clone --filter=blob:none --quiet https://github.com/evfro/polara.git /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-u4mjxjc2/polara_65159b1df5d349e4933241282b5f60d9
  Resolved https://github.com/evfro/polara.git to commit 86a5d247335242f31baf8fb68e472b651ae6770f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [67]:
# import sys
# sys.path.append('/kaggle/working/Actions-Speak-Louder-than-Words-research')

In [68]:
from typing import Callable, Dict, List, Tuple, Any, Optional
from collections import defaultdict


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch.nn.functional as F

from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix


from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)
from src.data.loaders import (load_ml20m, load_amzn_books, load_yambda, split_and_reindex)
from src.data.evaluation import (evaluate, evaluate_model)
from src.data.utils import (create_masked_tensor, build_q_from_train_targets)

In [69]:
class TransfromerTrainDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        self.samples = []

        for uid, history in histories.items():
            indicies = np.arange(0, len(history), max_seq_len)
            splits = np.split(history[:-1], indicies)[1:]
            targets = np.split(history, indicies+1)[1:]
            
            self.samples.extend(
                {
                    'uid': uid,
                    'history': list(s),
                    'targets': list(t),
                    'length': len(s)
                }
                for s, t in zip(splits, targets) if len(s) > 0 and len(t) > 0
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [70]:
class TransfromerEvalDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        targets: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        
        targets_uids = targets.keys()
        self.samples = [
            {
                'uid': uid,
                'history': history[-max_seq_len:],
                'length': len(history[-max_seq_len:])
            }
            for uid, history in histories.items() if uid in targets_uids and history
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [71]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:

    uids = torch.stack([torch.tensor(b['uid'], dtype=torch.long) for b in batch])
    lengths = torch.stack([torch.tensor(b['length'], dtype=torch.long) for b in batch])
    histories = torch.concat([torch.tensor(b['history'], dtype=torch.long) for b in batch])

    res = {
        'uid': uids,
        'length': lengths,
        'history': histories
        }

    if batch[0].get('targets', None):
        targets = torch.concat([torch.tensor(b['targets'], dtype=torch.long) for b in batch])
        res['targets'] = targets

    return res

In [72]:
# # !gdown --id 1KzSOO2huM1e8ZlEOa4QzVV-TtW70fmeV -O ml-20m.zip
# # !gdown --id 1tyKvtBnkDV8hwkx1s3BgT3LoH-s5DA1j -O amzn-books-20m.csv.gz
# !gdown --id 11WLpAlB7eQTgwrCQSEJtOIe-qTe6qxuC -O yambda-10m.parquet

In [73]:
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
DEVICE='cuda'

config = {
    # "ml": {
    #     "max_seq_len": 200,
    #     "test_quantile": 0.1,
    #     "interactions_path": "ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
    #     "col_mapping": {
    #         "userid": "user_id",
    #         "movieid": "item_id",
    #         "rating": "feedback",
    #         "timestamp": "timestamp",
    #     },
    # },
    "yambda": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "yambda-10m.parquet", # из ноута Кирилла (мб кто подкинет ссылку)
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    # "amzn-books": {
    #     "max_seq_len": 50,
    #     "test_quantile": 0.1,
    #     "interactions_path": "amzn-books-20m.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
    #     "col_mapping": {
    #         "user_id": "user_id",
    #         "parent_asin": "item_id",
    #         "rating": "feedback",
    #         "timestamp": "timestamp",
    #     },
    # },
}

In [74]:
# ml_df = load_ml20m(config["ml"]["interactions_path"], config=config["ml"])
yambda_df = load_yambda(config["yambda"]["interactions_path"], config=config["yambda"])
# amzn_df = load_amzn_books(config["amzn-books"]["interactions_path"], config=config["amzn-books"])

In [75]:
# ml_train, ml_test, ml_data_description = split_and_reindex(ml_df, config=config["ml"])
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda"])
# amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [76]:
datasets = {
    # "ml": {
    #     "train": ml_train,
    #     "test": ml_test,
    #     "desc": ml_data_description,
    # },
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    },
    # "amzn-books": {
    #     "train": amzn_train,
    #     "test": amzn_test,
    #     "desc": amzn_data_description,
    # },
}

In [77]:
def build_histories(df: pd.DataFrame, desc: Dict) -> Dict:
    return df.sort_values([desc['users'], desc['timestamp']], ascending=True).groupby(desc['users'])[desc['items']].apply(list).to_dict()

for name, d in datasets.items():
    train, test, desc = d['train'], d['test'], d['desc']
    max_seq_len = config[name]['max_seq_len']

    histories = build_histories(train, desc)
    targets = build_histories(test, desc)
    
    d['histories'] = histories
    d['targets'] = targets  

    d['train_dataset'] = TransfromerTrainDataset(histories=histories, max_seq_len=max_seq_len)
    d['eval_dataset'] = TransfromerEvalDataset(histories=histories, targets=targets, max_seq_len=max_seq_len)

    d['train_dataloader'] = DataLoader(
        dataset=d['train_dataset'],
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        drop_last=True
    )
    d['eval_dataloader'] = DataLoader(
        dataset=d['eval_dataset'],
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        drop_last=False
    )

In [78]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,yambda,7473676,748537,8222213,0.909,0.091,79059,53927,163448


TopPop

In [79]:
class TopPopRec:
    def __init__(self, n_items: int):
        self.n_items = n_items
        self.popular_items = None

    def fit(self, train_histories: Dict[int, List[int]]):
        all_items = np.concatenate(list(train_histories.values()))
        counts = np.bincount(all_items, minlength=self.n_items)
        self.popular_items = np.argsort(counts)[::-1].tolist()
        return self

    def predict(self, user_ids: List[int], topk: int = 100) -> Dict[int, List[int]]:
        top = self.popular_items[:topk]
        return dict(zip(user_ids, [top] * len(user_ids)))

RandomRec

In [80]:
class RandomRec:
    def __init__(self, n_items: int):
        self.n_items = n_items

    def fit(self, train_histories: Dict[int, List[int]]):
        return self

    def predict(self, user_ids: List[int], topk: int = 100) -> Dict[int, List[int]]:
        rng = np.random.default_rng()
        return {uid: rng.choice(self.n_items, size=topk, replace=False).tolist() for uid in user_ids}

iALS

In [81]:
class iALSRec:
    def __init__(self, n_items: int, factors: int = 64, iterations: int = 20):
        self.n_items = n_items
        self.model = AlternatingLeastSquares(
            factors=factors,
            iterations=iterations,
            use_gpu=False
        )
        self.user_factors = None
        self.item_factors = None

    def fit(self, train_histories: Dict[int, List[int]]):
        n_users = max(train_histories.keys()) + 1
        rows = np.repeat(list(train_histories.keys()), [len(v) for v in train_histories.values()])
        cols = np.concatenate(list(train_histories.values()))
        data = np.ones(len(rows))

        user_item = csr_matrix((data, (rows, cols)), shape=(n_users, self.n_items))
        item_user = user_item.T.tocsr()
        self.model.fit(item_user)

        self.user_factors = self.model.user_factors
        self.item_factors = self.model.item_factors
        return self

    def predict(self, user_ids: List[int], topk: int = 100, batch_size: int = 1000) -> Dict[int, List[int]]:
        result = {}
        for i in range(0, len(user_ids), batch_size):
            batch_uids = user_ids[i:i + batch_size]
            user_vecs = self.user_factors[batch_uids]
            scores = user_vecs @ self.item_factors.T
            # argpartition находит топ-K без полной сортировки
            top_k_idx = np.argpartition(-scores, kth=topk, axis=1)[:, :topk]
            # сортируем только топ-K для правильного порядка
            top_k_scores = scores[np.arange(len(batch_uids))[:, None], top_k_idx]
            sorted_idx = np.argsort(-top_k_scores, axis=1)
            top_indices = top_k_idx[np.arange(len(batch_uids))[:, None], sorted_idx]
            result.update(dict(zip(batch_uids, top_indices.tolist())))
        return result

SASRec

In [82]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.dropout = dropout

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.n_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, n_heads, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)


    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:

        x = self.qkv(x)

        Q = self.split_heads(x[:, :, :self.d_model])
        K = self.split_heads(x[:, :, self.d_model:2*self.d_model])
        V = self.split_heads(x[:, :, 2*self.d_model:])

        causal_mask = torch.ones(Q.size(-2), K.size(-2), dtype=torch.bool, device=Q.device).tril(diagonal=0).unsqueeze(0).unsqueeze(0)
        padding_mask = mask.unsqueeze(1).unsqueeze(1)
        attn_mask = causal_mask & padding_mask

        attn_output = F.scaled_dot_product_attention(Q, K, V, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.out(self.combine_heads(attn_output))

In [83]:
class MLP(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.0):
        super().__init__()
        self.linear = nn.Linear(d_model, 4 * d_model)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(4 * d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, d_model = x.size()

        x = self.linear(x)
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.out(x)

        return x

In [84]:
class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()

        self.dropout = dropout

        self.ln1 = nn.LayerNorm(d_model)
        self.csa = CausalSelfAttention(d_model, n_heads, dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:

        B, S, D = x.size()

        if mask is None:
            mask = torch.ones(B, S, dtype=torch.bool, device=x.device)

        x1 = self.ln1(x)
        x1 = self.csa(x1, mask)
        x1 = self.dropout1(x1)

        x = x + x1

        x2 = self.ln2(x)
        x2 = self.mlp(x2)
        x2 = self.dropout2(x2)

        x = x + x2


        return x

In [85]:
class GPT(nn.Module):
    def __init__(
        self,
        max_seq_len: int,
        n_layers: int,
        d_model: int,
        n_heads: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.dropout = dropout
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:

        x = self.drop(x)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln(x)

        return x

In [86]:
class UserEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embedding_dim)
        self.GPT = GPT(max_seq_len, n_layers, embedding_dim, n_heads, dropout)



    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:

        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        item_emb = self.item_embeddings(hist)
        pos_emb = self.pos_emb(torch.arange(hist.size(1), dtype=torch.long, device=hist.device))
        x = item_emb + pos_emb
        x[~mask] = 0.0
        x = self.GPT(x, mask)

        return x[mask]

In [87]:
class TrainSASRecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        
        self.encoder = UserEncoder(
            num_items=num_items,
            embedding_dim=embedding_dim,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            n_heads=n_heads,
            dropout=dropout,
        )
        self.num_negatives = num_negatives
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer("q_counts", q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if "weight" in key:
                nn.init.trunc_normal_(
                    value.data,
                    std=initializer_range,
                    a=-2 * initializer_range,
                    b=2 * initializer_range,
                )
            elif "bias" in key:
                nn.init.zeros_(value.data)


    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        return self.compute_loss(encoder_output, inputs)


    def compute_loss(
        self, encoder_output: torch.Tensor, inputs: Dict[str, Any]
    ) -> torch.Tensor:

        target_ids = inputs['targets']
        n = target_ids.shape[0]

        negative_pos = torch.randint(0, n, (n, self.num_negatives), device=target_ids.device)
        negative_ids = target_ids[negative_pos]

        candidate_ids = torch.cat([target_ids[:, None], negative_ids], dim=1)
        candidate_emb = self.encoder.item_embeddings(candidate_ids)

        logits = (encoder_output[:, None, :] * candidate_emb).sum(dim=-1)

        log_total = torch.log(self.q_counts.sum().clamp_min(self.eps))
        log_q = torch.log(self.q_counts[negative_ids].clamp_min(self.eps)) - log_total

        correction = torch.cat([torch.zeros(n, 1, device=logits.device), log_q], dim=1)
        logits = logits - correction

        labels = torch.zeros(n, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [88]:
def train(
    dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    device: str = "cuda",
) -> tuple[dict[str, Any], list[float]]:

    model.train()
    epoch_losses = []

    for epoch in tqdm(range(num_epochs), desc="Epochs"):
        total_loss = 0.0
        num_batches = 0

        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            loss = model(batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        epoch_loss = total_loss / num_batches
        epoch_losses.append(epoch_loss)
        tqdm.write(f"Epoch {epoch+1} loss: {epoch_loss:.4f}")

    return model.state_dict(), epoch_losses

In [89]:
class EvalSASRecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        
        self.encoder = UserEncoder(
            num_items=num_items,
            embedding_dim=embedding_dim,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            n_heads=n_heads,
            dropout=dropout
        )


    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        lengths = inputs['length']
        last_idx = lengths.cumsum(dim=0) - 1
        user_emb = encoder_output[last_idx]

        item_emb = self.encoder.item_embeddings.weight
        scores = user_emb @ item_emb.T

        return scores

GRU4Rec

In [90]:
class GRUEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        n_layers: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=embedding_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        x = self.dropout(self.item_embeddings(hist))

        lengths_cpu = inputs['length'].cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
        output, _ = self.gru(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(output, batch_first=True)

        return output[mask]

In [91]:
class TrainGRU4RecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        n_layers: int = 2,
        dropout: float = 0.1,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        self.encoder = GRUEncoder(num_items, embedding_dim, n_layers, dropout)
        self.num_negatives = num_negatives
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer('q_counts', q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if 'weight' in key:
                nn.init.trunc_normal_(value.data, std=initializer_range, a=-2*initializer_range, b=2*initializer_range)
            elif 'bias' in key:
                nn.init.zeros_(value.data)

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        return self.compute_loss(encoder_output, inputs)

    def compute_loss(self, encoder_output: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        target_ids = inputs['targets']
        n = target_ids.shape[0]

        negative_pos = torch.randint(0, n, (n, self.num_negatives), device=target_ids.device)
        negative_ids = target_ids[negative_pos]

        candidate_ids = torch.cat([target_ids[:, None], negative_ids], dim=1)
        candidate_emb = self.encoder.item_embeddings(candidate_ids)

        logits = (encoder_output[:, None, :] * candidate_emb).sum(dim=-1)

        log_total = torch.log(self.q_counts.sum().clamp_min(self.eps))
        log_q = torch.log(self.q_counts[negative_ids].clamp_min(self.eps)) - log_total
        correction = torch.cat([torch.zeros(n, 1, device=logits.device), log_q], dim=1)
        logits = logits - correction

        labels = torch.zeros(n, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [92]:
class EvalGRU4RecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        n_layers: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        
        self.encoder = GRUEncoder(num_items, embedding_dim, n_layers, dropout)

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        lengths = inputs['length']
        last_idx = lengths.cumsum(dim=0) - 1
        
        user_emb = encoder_output[last_idx]
        item_emb = self.encoder.item_embeddings.weight
        
        return user_emb @ item_emb.T

BERT4Rec

In [93]:
class BERTEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 200,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.mask_token_id = num_items
        self.item_embeddings = nn.Embedding(num_items + 1, embedding_dim) # +1 для [MASK]
        self.pos_emb = nn.Embedding(max_seq_len + 1, embedding_dim) # +1 для [MASK] позиционного энкодера
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=n_heads,
            dim_feedforward=4 * embedding_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers, enable_nested_tensor=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        item_emb = self.item_embeddings(hist)
        pos_emb = self.pos_emb(torch.arange(hist.size(1), dtype=torch.long, device=hist.device))
        x = self.dropout(item_emb + pos_emb)
        x[~mask] = 0.0
        pad_mask = ~mask
        x = self.transformer(x, src_key_padding_mask=pad_mask)
        return x, mask

In [94]:
class TrainBERT4RecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        max_seq_len: int = 200,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
        mask_prob: float = 0.2,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        self.encoder = BERTEncoder(num_items, embedding_dim, max_seq_len, n_layers, n_heads, dropout)
        self.num_negatives = num_negatives
        self.mask_prob = mask_prob
        self.mask_token_id = num_items
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer('q_counts', q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if 'weight' in key:
                nn.init.trunc_normal_(value.data, std=initializer_range, a=-2*initializer_range, b=2*initializer_range)
            elif 'bias' in key:
                nn.init.zeros_(value.data)

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
    
        rand = torch.rand(hist.shape, device=hist.device)
        masked_positions = (rand < self.mask_prob) & mask
        no_mask_rows = ~masked_positions.any(dim=1)
        last_valid = mask.sum(dim=1) - 1
        masked_positions[no_mask_rows, last_valid[no_mask_rows]] = True
    
        masked_hist = hist.clone()
        masked_hist[masked_positions] = self.mask_token_id
    
        masked_flat = masked_hist[mask]
        modified_inputs = {'history': masked_flat, 'length': inputs['length']}
    
        encoder_output, out_mask = self.encoder(modified_inputs)
    
        target_ids = hist[masked_positions]
        encoder_output_masked = encoder_output[masked_positions]
    
        return self.compute_loss(encoder_output_masked, target_ids)

    def compute_loss(self, encoder_output: torch.Tensor, target_ids: torch.Tensor) -> torch.Tensor:
        n = target_ids.shape[0]
        negative_pos = torch.randint(0, n, (n, self.num_negatives), device=target_ids.device)
        negative_ids = target_ids[negative_pos]

        candidate_ids = torch.cat([target_ids[:, None], negative_ids], dim=1)
        candidate_emb = self.encoder.item_embeddings(candidate_ids)

        logits = (encoder_output[:, None, :] * candidate_emb).sum(dim=-1)

        log_total = torch.log(self.q_counts.sum().clamp_min(self.eps))
        log_q = torch.log(self.q_counts[negative_ids].clamp_min(self.eps)) - log_total
        correction = torch.cat([torch.zeros(n, 1, device=logits.device), log_q], dim=1)
        logits = logits - correction

        labels = torch.zeros(n, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [95]:
class EvalBERT4RecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 200,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.encoder = BERTEncoder(num_items, embedding_dim, max_seq_len, n_layers, n_heads, dropout)
        self.mask_token_id = num_items

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        B = hist.size(0)

        
        mask_col = torch.full((B, 1), self.mask_token_id, dtype=torch.long, device=hist.device) # добавляем [MASK] в конец каждой истории
        hist_with_mask = torch.cat([hist, mask_col], dim=1)
        mask_with_mask = torch.cat([mask, torch.ones(B, 1, dtype=torch.bool, device=hist.device)], dim=1)

        new_lengths = inputs['length'] + 1
        flat = hist_with_mask[mask_with_mask]
        modified_inputs = {'history': flat, 'length': new_lengths}

        encoder_output, new_mask = self.encoder(modified_inputs)

        
        last_idx = new_lengths.cumsum(dim=0) - 1 # берём скрытое состояние последней позиции ([MASK])
        flat_output = encoder_output[new_mask]
        user_emb = flat_output[last_idx]

        item_emb = self.encoder.item_embeddings.weight[:self.mask_token_id]  # только реальные айтемы
        
        return user_emb @ item_emb.T

In [ ]:
results = []

for name, d in datasets.items():
    desc = d['desc']
    user_ids = list(d['targets'].keys())

    # TopPopRec
    toppop_model = TopPopRec(n_items=desc['n_items']).fit(d['histories'])
    for topk in [10, 50, 200]:
        toppop_recs = toppop_model.predict(user_ids, topk=topk)
        toppop_metrics = evaluate(targets=d['targets'], candidates=toppop_recs, catalog_size=desc['n_items'], topk=topk)
        print(f"[{name}] TopPop @{topk}: {toppop_metrics}")
        results.append({'dataset': name, 'model': 'toppop', 'topk': topk, **toppop_metrics})

    # RandomRec
    random_model = RandomRec(n_items=desc['n_items']).fit(d['histories'])
    for topk in [10, 50, 200]:
        random_recs = random_model.predict(user_ids, topk=topk)
        random_metrics = evaluate(targets=d['targets'], candidates=random_recs, catalog_size=desc['n_items'], topk=topk)
        print(f"[{name}] Random @{topk}: {random_metrics}")
        results.append({'dataset': name, 'model': 'random', 'topk': topk, **random_metrics})

    # iALS
    ials_model = iALSRec(n_items=desc['n_items'], factors=64, iterations=20).fit(d['histories'])
    for topk in [10, 50, 200]:
        ials_recs = ials_model.predict(user_ids, topk=topk)
        ials_metrics = evaluate(targets=d['targets'], candidates=ials_recs, catalog_size=desc['n_items'], topk=topk)
        print(f"[{name}] iALS @{topk}: {ials_metrics}")
        results.append({'dataset': name, 'model': 'ials', 'topk': topk, **ials_metrics})

    # q_counts

    train_target_ids = torch.tensor([target for idx in range(len(d['train_dataset'])) for target in d['train_dataset'][idx]['targets']], dtype=torch.long)
    q_counts = build_q_from_train_targets(train_target_ids, catalog_size=desc['n_items'])


    # SASRec

    train_model = TrainSASRecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        num_negatives=512,
        q_counts=q_counts,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
        dropout=0.1,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(train_model.parameters(), lr=1e-3)

    checkpoint, losses = train(
        dataloader=d['train_dataloader'],
        model=train_model,
        optimizer=optimizer,
        num_epochs=10,
        device=DEVICE,
    )

    eval_model = EvalSASRecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
    ).to(DEVICE)

    encoder_state = {
        key.removeprefix("encoder."): value
        for key, value in checkpoint.items()
        if key.startswith("encoder.")
    }
    eval_model.encoder.load_state_dict(encoder_state)

    for topk in [10, 50, 200]:
        metrics = evaluate_model(dataloader=d['eval_dataloader'], model=eval_model, catalog_size=desc['n_items'], topk=topk, device=DEVICE, targets=d['targets'], evaluate_fn=evaluate)
        print(f"[{name}] SASRec @{topk}: {metrics}")
        results.append({'dataset': name, 'model': 'sasrec', 'topk': topk, **metrics})

    # GRU4Rec
    train_gru = TrainGRU4RecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        num_negatives=512,
        q_counts=q_counts,
        n_layers=2,
        dropout=0.1,
    ).to(DEVICE)
    optimizer_gru = torch.optim.Adam(train_gru.parameters(), lr=1e-3)
    checkpoint_gru, losses_gru = train(
        dataloader=d['train_dataloader'],
        model=train_gru,
        optimizer=optimizer_gru,
        num_epochs=10,
        device=DEVICE,
    )
    eval_gru = EvalGRU4RecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        n_layers=2,
    ).to(DEVICE)
    encoder_state_gru = {
        key.removeprefix('encoder.'): value
        for key, value in checkpoint_gru.items()
        if key.startswith('encoder.')
    }
    eval_gru.encoder.load_state_dict(encoder_state_gru)
    
    for topk in [10, 50, 200]:
        metrics_gru = evaluate_model(dataloader=d['eval_dataloader'], model=eval_gru, catalog_size=desc['n_items'], topk=topk, device=DEVICE, targets=d['targets'], evaluate_fn=evaluate)
        print(f"[{name}] GRU4Rec @{topk}: {metrics_gru}")
        results.append({'dataset': name, 'model': 'gru4rec', 'topk': topk, **metrics_gru})

    # BERT4Rec
    train_bert = TrainBERT4RecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        num_negatives=512,
        q_counts=q_counts,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
        dropout=0.1,
        mask_prob=0.2,
    ).to(DEVICE)

    optimizer_bert = torch.optim.Adam(train_bert.parameters(), lr=1e-3)
    checkpoint_bert, losses_bert = train(
        dataloader=d['train_dataloader'],
        model=train_bert,
        optimizer=optimizer_bert,
        num_epochs=10,
        device=DEVICE,
    )

    eval_bert = EvalBERT4RecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
    ).to(DEVICE)

    encoder_state_bert = {
        key.removeprefix('encoder.'): value
        for key, value in checkpoint_bert.items()
        if key.startswith('encoder.')
    }
    eval_bert.encoder.load_state_dict(encoder_state_bert)

    for topk in [10, 50, 200]:
        metrics_bert = evaluate_model(dataloader=d['eval_dataloader'], model=eval_bert, catalog_size=desc['n_items'], topk=topk, device=DEVICE, targets=d['targets'], evaluate_fn=evaluate)
        print(f"[{name}] BERT4Rec @{topk}: {metrics_bert}")
        results.append({'dataset': name, 'model': 'bert4rec', 'topk': topk, **metrics_bert})

pd.DataFrame(results)

[yambda] TopPop @10: defaultdict(<class 'float'>, {'hitrate': 0.04363305950636972, 'recall': np.float64(0.00969571298766336), 'ndcg': np.float64(0.007205607685358081), 'coverage': 6.118153785913563e-05})
[yambda] TopPop @50: defaultdict(<class 'float'>, {'hitrate': 0.1414875665251173, 'recall': np.float64(0.025632086941975125), 'ndcg': np.float64(0.012553901746647056), 'coverage': 0.0003059076892956781})
[yambda] TopPop @200: defaultdict(<class 'float'>, {'hitrate': 0.2786730209357094, 'recall': np.float64(0.05864600501945935), 'ndcg': np.float64(0.02145386446752003), 'coverage': 0.0012236307571827124})
[yambda] Random @10: defaultdict(<class 'float'>, {'hitrate': 0.0005933947744172677, 'recall': np.float64(8.360067004159947e-05), 'ndcg': np.float64(7.44308561771119e-05), 'coverage': 0.9630279966717243})
[yambda] Random @50: defaultdict(<class 'float'>, {'hitrate': 0.003819978860311161, 'recall': np.float64(0.0002609489621470269), 'ndcg': np.float64(0.0001453750586372549), 'coverage': 